## Основные функции парсера


In [1]:
!pip -q install gspread pandas

In [2]:
import re
import pandas as pd
import gspread

try:
    from google.colab import auth, files
    import google.auth
    IN_COLAB = True
except Exception:
    auth = None
    files = None
    google = None
    IN_COLAB = False


# ==== CONSTANTS ====
DAYS = ['Понедельник', 'Вторник', 'Среда', 'Четверг', 'Пятница', 'Суббота']
DAY_BLOCKS = {
    'Понедельник': (2, 13),
    'Вторник': (15, 26),
    'Среда': (28, 39),
    'Четверг': (41, 52),
    'Пятница': (54, 65),
    'Суббота': (67, 78),
}
DAY_ORDER = {d: i for i, d in enumerate(DAYS)}

PAIR_TIMES_MON_FRI = {
    1: '08:30–10:00',
    2: '10:10–11:40',
    3: '12:20–13:50',
    4: '14:10–15:40',
    5: '15:50–17:20',
    6: '17:30–19:00',
}
PAIR_TIMES_SAT = {
    1: '08:30–10:00',
    2: '10:10–11:40',
    3: '11:50–13:20',
    4: '13:30–15:00',
    5: '15:10–16:40',
}

DISTANCE_ALIASES = {'эл', 'эл.', 'дист', 'дист.', 'дистанционно', 'онлайн', 'online'}


def get_gspread_client():
    if IN_COLAB:
        auth.authenticate_user()
        creds, _ = google.auth.default(
            scopes=['https://www.googleapis.com/auth/spreadsheets.readonly']
        )
        return gspread.authorize(creds)
    raise RuntimeError('Если запускаешь не в Colab, создай gc вручную и передай его в parse_teacher_schedule(..., gc=gc).')


def _norm(s: str) -> str:
    return re.sub(r'\s+', ' ', (s or '').strip().lower())


def pair_time(day_name: str, pair_num: int) -> str:
    if day_name == 'Суббота':
        return PAIR_TIMES_SAT.get(pair_num, '—')
    return PAIR_TIMES_MON_FRI.get(pair_num, '—')


def parse_day_and_date(cell_a: str):
    raw = (cell_a or '').strip()
    if not raw:
        return None, None
    parts = raw.split(maxsplit=1)
    day = parts[0].capitalize()
    date = parts[1].strip() if len(parts) > 1 else '—'
    if day not in DAY_ORDER:
        return None, None
    return day, date


def parse_cabinet(value: str, subject: str) -> str:
    raw = (value or '').strip()
    v = _norm(raw).replace('.', '')
    subj = _norm(subject)

    if v in DISTANCE_ALIASES:
        return 'Дистанционно'
    if not raw and 'практик' in subj:
        return 'Практика'
    return raw or '—'


def build_group_map(header_row):
    groups = []
    c = 2  # начиная с колонки C
    n = len(header_row)

    while c < n:
        cell = (header_row[c] or '').strip()
        if cell:
            group = cell
            subject_cols = [c, c + 1, c + 2]
            cab_col = c + 3 if c + 3 < n else None
            groups.append({
                'group': group,
                'subject_cols': subject_cols,
                'teacher_cols': subject_cols + ([cab_col] if cab_col is not None else []),
                'cab_col': cab_col,
            })
            c += 4
        else:
            c += 1

    return groups


def get_first_nonempty(row, cols):
    for c in cols:
        if c < len(row) and row[c]:
            return row[c].strip(), c
    return '', None


def parse_teacher_schedule(sheet_url, surname, sheet_titles=None, gc=None):
    if gc is None:
        gc = get_gspread_client()

    sh = gc.open_by_url(sheet_url)
    surname_n = _norm(surname)
    results = []
    seen = set()

    for ws in sh.worksheets():
        if ws.title == 'Звонки':
            continue
        if sheet_titles is not None and ws.title not in sheet_titles:
            continue

        data = ws.get_all_values()
        if len(data) < 4:
            continue

        header = data[1]  # строка 2
        groups = build_group_map(header)
        if not groups:
            continue

        for day_name, (start_row, end_row) in DAY_BLOCKS.items():
            if start_row >= len(data):
                continue

            day_cell = data[start_row][0] if len(data[start_row]) > 0 else ''
            parsed_day, day_date = parse_day_and_date(day_cell)
            day_final = parsed_day or day_name
            date_final = day_date or '—'

            pair_num = 1
            r = start_row
            while r <= min(end_row, len(data) - 2):
                row_subject = data[r]
                row_teacher = data[r + 1]

                pair_cell = row_subject[1].strip() if len(row_subject) > 1 else ''
                if pair_cell.isdigit():
                    pair_num = int(pair_cell)

                for gb in groups:
                    teacher_cell, _ = get_first_nonempty(row_teacher, gb['teacher_cols'])
                    if not teacher_cell or surname_n not in _norm(teacher_cell):
                        continue

                    subject, _ = get_first_nonempty(row_subject, gb['subject_cols'])
                    cab_raw = row_subject[gb['cab_col']].strip() if gb['cab_col'] is not None and gb['cab_col'] < len(row_subject) else ''
                    cabinet = parse_cabinet(cab_raw, subject)

                    item = {
                        'День': day_final,
                        'Дата': date_final,
                        'Пара': pair_num,
                        'Время': pair_time(day_final, pair_num),
                        'Курс': ws.title,
                        'Группа': gb['group'],
                        'Предмет': subject or '—',
                        'Кабинет': cabinet,
                        'Преподаватель': teacher_cell,
                    }

                    key = tuple(item.values())
                    if key not in seen:
                        seen.add(key)
                        results.append(item)

                pair_num += 1
                r += 2

    results.sort(key=lambda r: (
        DAY_ORDER.get(r['День'], 99),
        r['Дата'],
        r['Пара'],
        r['Курс'],
        r['Группа']
    ))
    return results


def format_by_days(rows):
    out = {}
    for r in rows:
        day_key = f"{r['День']} {r['Дата']}".strip()
        place = 'дистанционно' if r['Кабинет'] == 'Дистанционно' else f"каб {r['Кабинет']}"
        if r['Кабинет'] == 'Практика':
            place = 'практика'
        out.setdefault(day_key, []).append(
            f"Пара {r['Пара']} ({r['Время']}): {r['Курс']} | {r['Группа']} | {r['Предмет']} | {place}"
        )
    return out


def print_schedule_by_days(rows):
    out = format_by_days(rows)
    keys = sorted(out.keys(), key=lambda k: (DAY_ORDER.get(k.split(' ', 1)[0], 99), k.split(' ', 1)[1] if ' ' in k else ''))
    first = True
    for day_key in keys:
        if not first:
            print()
        first = False
        print(f"=== {day_key} ===")
        for line in out[day_key]:
            print(line)


def export_csv(rows, name='teacher_schedule.csv', download=False):
    df = pd.DataFrame(rows)
    df.to_csv(name, index=False, encoding='utf-8-sig')
    if download and files is not None:
        files.download(name)
    return name

In [3]:
def parse_full_schedule(sheet_url, sheet_titles=None, gc=None):
    """
    Читает всё расписание целиком по всем преподавателям.
    Возвращает list[dict] с полями:
    День, Дата, Пара, Время, Курс, Группа, Предмет, Кабинет, Преподаватель
    """
    if gc is None:
        gc = get_gspread_client()

    sh = gc.open_by_url(sheet_url)
    results = []
    seen = set()

    for ws in sh.worksheets():
        if ws.title == 'Звонки':
            continue
        if sheet_titles is not None and ws.title not in sheet_titles:
            continue

        data = ws.get_all_values()
        if len(data) < 4:
            continue

        header = data[1]
        groups = build_group_map(header)
        if not groups:
            continue

        for day_name, (start_row, end_row) in DAY_BLOCKS.items():
            if start_row >= len(data):
                continue

            day_cell = data[start_row][0] if len(data[start_row]) > 0 else ''
            parsed_day, day_date = parse_day_and_date(day_cell)
            day_final = parsed_day or day_name
            date_final = day_date or '—'

            pair_num = 1
            r = start_row
            while r <= min(end_row, len(data) - 2):
                row_subject = data[r]
                row_teacher = data[r + 1]

                pair_cell = row_subject[1].strip() if len(row_subject) > 1 else ''
                if pair_cell.isdigit():
                    pair_num = int(pair_cell)

                for gb in groups:
                    teacher_cell, _ = get_first_nonempty(row_teacher, gb['teacher_cols'])
                    subject, _ = get_first_nonempty(row_subject, gb['subject_cols'])

                    if not teacher_cell and not subject:
                        continue

                    cab_raw = row_subject[gb['cab_col']].strip() if gb['cab_col'] is not None and gb['cab_col'] < len(row_subject) else ''
                    cabinet = parse_cabinet(cab_raw, subject)

                    item = {
                        'День': day_final,
                        'Дата': date_final,
                        'Пара': pair_num,
                        'Время': pair_time(day_final, pair_num),
                        'Курс': ws.title,
                        'Группа': gb['group'],
                        'Предмет': subject or '—',
                        'Кабинет': cabinet,
                        'Преподаватель': teacher_cell or '—',
                    }

                    key = tuple(item.values())
                    if key not in seen:
                        seen.add(key)
                        results.append(item)

                pair_num += 1
                r += 2

    results.sort(key=lambda r: (
        DAY_ORDER.get(r['День'], 99),
        r['Дата'],
        r['Пара'],
        r['Курс'],
        r['Группа'],
        r['Преподаватель']
    ))
    return results


def analyze_cabinet_conflicts_full(all_rows):
    """
    Ищет реальные конфликты по кабинетам по всему расписанию:
    ошибка только если в один и тот же день/дату/пару/кабинет
    одновременно стоят И РАЗНЫЕ группы, И РАЗНЫЕ преподаватели.

    Исключения:
    - разные группы, но один преподаватель -> не ошибка;
    - если все предметы в конфликтующем наборе относятся к физической культуре -> не ошибка.
    """
    buckets = {}

    for r in all_rows:
        cabinet = (r.get('Кабинет') or '').strip()
        subject = (r.get('Предмет') or '').strip()
        teacher = (r.get('Преподаватель') or '').strip()
        group = (r.get('Группа') or '').strip()

        if not cabinet or cabinet in {'—', 'Дистанционно', 'Практика'}:
            continue
        if not subject or subject == '—':
            continue
        if not group or group == '—':
            continue
        if not teacher or teacher == '—':
            continue

        key = (
            r.get('День', '—'),
            r.get('Дата', '—'),
            r.get('Пара', '—'),
            r.get('Время', '—'),
            cabinet,
        )
        buckets.setdefault(key, []).append(r)

    conflicts = []
    for key, items in buckets.items():
        groups = sorted({(x.get('Группа') or '—').strip() for x in items if (x.get('Группа') or '').strip() and x.get('Группа') != '—'})
        teachers = sorted({(x.get('Преподаватель') or '—').strip() for x in items if (x.get('Преподаватель') or '').strip() and x.get('Преподаватель') != '—'})
        courses = sorted({(x.get('Курс') or '—').strip() for x in items if (x.get('Курс') or '').strip()})
        subjects = sorted({(x.get('Предмет') or '—').strip() for x in items if (x.get('Предмет') or '').strip() and x.get('Предмет') != '—'})

        group_conflict = len(groups) > 1
        teacher_conflict = len(teachers) > 1

        if not (group_conflict and teacher_conflict):
            continue

        subject_norms = [(' '.join((s or '').lower().split())) for s in subjects]
        all_physical = bool(subject_norms) and all('физическ' in s and 'культур' in s for s in subject_norms)
        if all_physical:
            continue

        conflicts.append({
            'День': key[0],
            'Дата': key[1],
            'Пара': key[2],
            'Время': key[3],
            'Кабинет': key[4],
            'Курсы': '; '.join(courses) if courses else '—',
            'Группы': '; '.join(groups) if groups else '—',
            'Преподаватели': '; '.join(teachers) if teachers else '—',
            'Предметы': ' | '.join(subjects) if subjects else '—',
            'Количество_записей': len(items),
            'Тип_конфликта': 'Разные группы и разные преподаватели',
        })

    conflicts.sort(key=lambda r: (
        DAY_ORDER.get(r['День'], 99),
        r['Дата'],
        r['Пара'],
        r['Кабинет']
    ))
    return conflicts


def print_cabinet_conflicts_full(all_rows):
    conflicts = analyze_cabinet_conflicts_full(all_rows)
    if not conflicts:
        print('Конфликтов по кабинетам во всём расписании не найдено')
        return conflicts

    current_day = None
    for c in conflicts:
        day_key = f"{c['День']} {c['Дата']}"
        if day_key != current_day:
            if current_day is not None:
                print()
            current_day = day_key
            print(f'=== {day_key} ===')

        print(
            f"Пара {c['Пара']} ({c['Время']}), каб {c['Кабинет']}: "
            f"{c['Тип_конфликта']} | курсы: {c['Курсы']} | группы: {c['Группы']} | "
            f"преподаватели: {c['Преподаватели']} | предметы: {c['Предметы']}"
        )

    return conflicts


def export_full_conflicts_csv(all_rows, name='cabinet_conflicts_full.csv', download=False):
    conflicts = analyze_cabinet_conflicts_full(all_rows)
    df = pd.DataFrame(conflicts)
    df.to_csv(name, index=False, encoding='utf-8-sig')
    if download and 'files' in globals() and files is not None:
        files.download(name)
    return name

## Моё расписание

In [4]:
# ==== RUN ====
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1X6YF54l1rgP7MFfkTa1b_L6f4f3aWuADZwF8wwTWKK4/edit?gid=1584915456#gid=1584915456'
SURNAME = 'Белоусов'
rows = parse_teacher_schedule(SHEET_URL, SURNAME)
print_schedule_by_days(rows)
#export_csv(rows)

## Расписания других преподавателей

In [5]:
SURNAME = 'Коробинцева'
rows = parse_teacher_schedule(SHEET_URL, SURNAME)
print_schedule_by_days(rows)

=== Среда 02.09 ===
Пара 1 (08:30–10:00): 1 курс | ДОУ-104 | Информатика | каб 32
Пара 2 (10:10–11:40): 1 курс | СЗС-134 | Информатика | каб 32
Пара 3 (12:20–13:50): 1 курс | ПКП-123 | Информатика | каб 32
Пара 4 (14:10–15:40): 4 курс | СЗС-431 | Системы автоматизированного проектирования | каб 32
Пара 5 (15:50–17:20): 3 курс | ЮСП-301 | Информационные технологии в юридической деятельности | дистанционно
Пара 5 (15:50–17:20): 4 курс | СЗС-431 | МДК.06.02 Информационное моделирование | каб 32
Пара 6 (17:30–19:00): 3 курс | ЮСП-301 | Информационные технологии в юридической деятельности | дистанционно

=== Четверг 03.09 ===
Пара 1 (08:30–10:00): 1 курс | СЗС-134 | Информатика | дистанционно
Пара 2 (10:10–11:40): 1 курс | ДОУ-104 | Информатика | дистанционно
Пара 5 (15:50–17:20): 3 курс | ЮСП-301 | Информационные технологии в юридической деятельности | дистанционно
Пара 6 (17:30–19:00): 3 курс | ЮСП-301 | Информационные технологии в юридической деятельности | дистанционно

=== Пятница 04.0

In [6]:
SURNAME = 'Лоренц'
rows = parse_teacher_schedule(SHEET_URL, SURNAME)
print_schedule_by_days(rows)

=== Среда 02.09 ===
Пара 1 (08:30–10:00): 1 курс | ТОС-105 | Физика | каб 15
Пара 2 (10:10–11:40): 1 курс | АВМ-112 | Физика | каб 15
Пара 3 (12:20–13:50): 1 курс | ЭМР-111 | Физика | каб 15
Пара 4 (14:10–15:40): 2 курс | ЭМР-207 | Электротехника с основами электроники | каб 15
Пара 5 (15:50–17:20): 2 курс | ЭМР-207 | Электротехника с основами электроники | каб 15

=== Четверг 03.09 ===
Пара 1 (08:30–10:00): 1 курс | ТИС-102 | Физика | каб 15
Пара 2 (10:10–11:40): 1 курс | ЭМР-111 | Физика | каб 15
Пара 3 (12:20–13:50): 1 курс | ЭМР-112 | Физика | каб 15
Пара 4 (14:10–15:40): 2 курс | ЭМР-208 | Электротехника с основами электроники | каб 15
Пара 5 (15:50–17:20): 2 курс | ЭМР-208 | Электротехника с основами электроники | каб 15

=== Пятница 04.09 ===
Пара 1 (08:30–10:00): 1 курс | ЭМР-111 | Охрана труда | каб 15
Пара 2 (10:10–11:40): 1 курс | ЭМР-111 | Электротехника с основами электроники | каб 15
Пара 3 (12:20–13:50): 1 курс | ЭМР-111 | МДК 01.01 Технология электромонтажных и сборочны

In [7]:
SURNAME = 'Барнякова'
rows = parse_teacher_schedule(SHEET_URL, SURNAME)
print_schedule_by_days(rows)

=== Среда 02.09 ===
Пара 1 (08:30–10:00): 1 курс | ТИС-102 | Русский язык | каб 21
Пара 2 (10:10–11:40): 1 курс | ТМС-104 | Русский язык | каб 21
Пара 3 (12:20–13:50): 1 курс | СЗС-134 | Русский язык | каб 21
Пара 4 (14:10–15:40): 2 курс | ТГО-201 | МДК.01.03 Соблюдение норм этики делового общения | каб 21
Пара 5 (15:50–17:20): 3 курс | ПКП-320 | Деловая культура и антикоррупционное поведение | каб 21

=== Четверг 03.09 ===
Пара 1 (08:30–10:00): 1 курс | ПКП-123 | Русский язык | каб 21
Пара 2 (10:10–11:40): 1 курс | ТИС-101 | Русский язык | каб 21
Пара 3 (12:20–13:50): 1 курс | ДОУ-104 | Русский язык | каб 21
Пара 4 (14:10–15:40): 3 курс | ПКП-320 | Деловая культура и антикоррупционное поведение | каб 21
Пара 5 (15:50–17:20): 3 курс | ПКП-319 | Деловая культура и антикоррупционное поведение | каб 21

=== Пятница 04.09 ===
Пара 1 (08:30–10:00): 1 курс | ПКП-123 | Русский язык | каб 21
Пара 2 (10:10–11:40): 1 курс | ТМС-104 | Русский язык | каб 21
Пара 3 (12:20–13:50): 1 курс | СЗС-134 |

## Ошибки кабинетов

In [8]:
# ==== NOTEBOOK CELLS ====
all_rows = parse_full_schedule(SHEET_URL)
conflicts = print_cabinet_conflicts_full(all_rows)
# export_full_conflicts_csv(all_rows, name='cabinet_conflicts_full.csv', download=False)
pd.DataFrame(conflicts)

Конфликтов по кабинетам во всём расписании не найдено


""
